# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daniakhan123/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## My Lane as an ML Task

**Lane:** Refresh / Content Opportunity Scoring

**Task type: Scoring / Ranking** (with a classification model underneath).

This is not a pure classification problem, even though I'll train a binary
classifier (declining vs. not) as the underlying model. The actual deliverable
is a **ranked queue** — an ordered list of pages a reviewer works through from
top to bottom, given limited review capacity. The classifier's output
probability is the ranking score; the final product is the order, not just
the label.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## Target or Proxy

**Starter proxy (Weeks 1-2):** `is_declining_label = (trend_direction == "down")`
— a same-window bucket, not a future outcome. Useful for learning the workflow,
but weak as a real target since it doesn't separate "declining recently" from
"about to decline."

**Stronger capstone target (what I'll move toward):** a future-window label —
features from the prior 90 days used to predict decline over the next 30 days:

## 3. Success metric

*One metric you can defend. What number means 'good'?*

## Success Metric

**Primary metric: Precision@50** (and Precision@20 as a secondary check).

Why not plain accuracy: the review team can only act on a limited number of
pages per cycle (their real capacity, not the whole inventory). Precision@K
directly answers "of the top K pages we'd tell someone to review, how many
are actually worth reviewing?" — which matches how the output gets used.
I'll also track Average Precision (whole-ranking quality) and compare against
the baseline hand-rule's Precision@50 = 0.240 as the bar to beat.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

import os, sys, subprocess
import pandas as pd, numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/daniakhan123/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Unit of analysis: one row = one content page (content_id),
# evaluated over its most recent 90-day window
cols_to_show = ["content_id", "client_id", "impressions_90d", "sessions_90d",
                 "content_age_days", "avg_position", "ctr", "trend_direction",
                 "is_declining_label"]
print("One row = one content page, over its trailing 90-day window")
print("Total rows (pages):", df.shape[0])
df[cols_to_show].head(10)

In [5]:
import os, sys, subprocess
import pandas as pd, numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/daniakhan123/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

cols_to_show = ["content_id", "client_id", "impressions_90d", "sessions_90d",
                 "content_age_days", "avg_position", "ctr", "trend_direction",
                 "is_declining_label"]
print("One row = one content page, over its trailing 90-day window")
print("Total rows (pages):", df.shape[0])
print(df[cols_to_show].head(10))

# Sketch of what the target column looks like: class balance
print("\n--- Target column ---")
print(df["is_declining_label"].value_counts())
print("\nDeclining rate:", round(df["is_declining_label"].mean(), 3))

One row = one content page, over its trailing 90-day window
Total rows (pages): 30000
             content_id          client_id  impressions_90d  sessions_90d  \
0  content_304f48230142  client_f369cb89fc             3803            17   
1  content_a1fb4e703a9e  client_4e07408562            15320             9   
2  content_9aa793d4d895  client_7f2253d7e2            12581            11   
3  content_331d6c4de07b  client_19581e27de            11751            78   
4  content_d99b7a2d90ca  client_3fdba35f04            19140           145   
5  content_d4084a4bc775  client_f369cb89fc             3970             5   
6  content_9a34b442b552  client_8722616204               20             1   
7  content_a63219c6e95a  client_19581e27de             1724            28   
8  content_5e6c160719bc  client_6208ef0f77            32574            68   
9  content_c27558df2b0c  client_19581e27de             1240             3   

   content_age_days  avg_position   ctr trend_direction  is_declin

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## Why ML Beats a Fixed Rule Here

A fixed rule (like the baseline: stale + visible + high impressions) can only
combine a handful of hand-picked thresholds a human guessed at, and it treats
every signal independently. Already-run evidence: the baseline rule scores
Precision@50 = 0.240, while a random forest reaches 0.740 on the same data —
roughly 3x more of the top 50 flagged pages are genuinely declining.

The reason: real decline is driven by *combinations* of signals (e.g. "high
impressions AND poor position AND stale AND low engagement" matters more than
any signal alone), and those combinations and their relative weights aren't
obvious to hand-tune. A learned model can pick up these interactions and
rank pages by evidence strength rather than pass/fail thresholds — while
still producing reason codes a reviewer can inspect, so it stays explainable,
not a black box.

This is not "ML for its own sake": the fixed rule is kept as the baseline to
beat, and the model is only worth using because it demonstrably beats it on
the metric that matches the real decision (Precision@K).

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-Check

- [x] Named the task type (scoring/ranking, backed by a classifier)
- [x] Named the target/proxy (starter: same-window decline label;
      capstone-bound: future-window decline label)
- [x] Named the success metric (Precision@50/20, matched to reviewer capacity)
- [x] Showed the unit of analysis as a real dataframe (one row = one page,
      over its 90-day window)
- [x] Explained why ML beats a fixed rule (3x Precision@50 lift, already
      observed, from combining signals a hand rule can't weigh jointly)
- [x] Tied output to a real action (ranked review queue a reviewer works
      through, capacity-limited)